# Meta-Learning: Learning to Learn

**Meta-learning** is about learning _how to learn_: instead of training a single policy for one
fixed goal, we train on a distribution of tasks so that - at _test time_ - the policy parameters
become a good **starting point** and **adapt quickly** to a new unseen task using only a small
amount of experience.

**Model-Agnostic Meta-Learning (MAML)** is a _gradient-based meta-learning_ algorithm that splits
experience into two parts:

- **Support (inner loop):** collect a little experience on a task and take a few gradient steps to
  get _adapted_ parameters $\theta'$.
- **Query (outer loop):** evaluate $\theta'$ on fresh experience from the same task, and update the
  _initial_ parameters $\theta$ so that adaptation improves performance.

So the meta-objective is: _choose $\theta$ such that a small amount of gradient-based adaptation
produces a big improvement on new tasks._

We will implement **MAML** (Model-Agnostic Meta-Learning) for RL using a simple policy-gradient
(REINFORCE) inner loop. We'll also implement **FOMAML** (First-Order MAML), and explicitly show
where the **second-order derivative** appears in full MAML. The overall implementation is
deliberately small and simplified.

> **Simplification:** We use vanilla policy gradient (REINFORCE) for both inner and outer loops.
> This is not the most sample-efficient choice, but it's the cleanest way to expose the mechanics of
> meta-learning.


In [ ]:
import copy

from contextlib import contextmanager, nullcontext
from dataclasses import dataclass
from typing import Dict, Tuple, List
import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Independent, Normal, TransformedDistribution
from torch.distributions.transforms import AffineTransform, TanhTransform

import matplotlib.pyplot as plt

from util.gymnastics import DEVICE, init_random

## The task: 2D point navigation with hidden goals

We'll train on a family of 2D navigation tasks:

- The agent controls a point in 2D.
- Each task corresponds to a **different hidden goal position**.
- The observation is just the current position (the goal is _not_ observed).
- The reward depends on the (hidden) goal.

So, without adaptation, the agent cannot know where to go. With meta-learning, the policy learns
parameters that can be adapted quickly using gradients from reward. We define a minimal environment:

- State: position $x_t \in \mathbb{R}^2$
- Action: delta $a_t \in \mathbb{R}^2$ (clipped)
- Dynamics: $x_{t+1} = x_t + a_t + \epsilon$ (optional noise)
- Each **task** is a different goal $g \in \mathbb{R}^2$ sampled from a distribution
- Reward: negative squared distance to goal: $r_t = -\lVert x_t - g \rVert^2$

The goal is **hidden** from the policy (the observation is only the position), so task
identification must come from reward signals and fast adaptation.


### Why does meta-learning work on this task?

The key insight is that the **goal is hidden**: the agent never observes the goal position directly.
Without adaptation, a policy has no way to know where to go — its best bet is to stay near the
origin (the "average" goal).

But the **reward** leaks information about the goal: a step that brings the agent closer to the
hidden goal yields a higher reward. So a few gradient steps on reward effectively "decode" which
direction the goal lies in, letting the policy specialize.

MAML exploits this by finding an initialization $\theta$ that is maximally sensitive to this reward
signal — one where a small gradient update reliably steers the policy toward the correct goal.


In [ ]:
@dataclass
class Point2DTask:
    goal: np.ndarray  # shape (2,)


class Point2DEnv:
    def __init__(self, horizon: int = 25, max_action: float = 0.15, noise_std: float = 0.00):
        self.horizon = horizon
        self.max_action = max_action
        self.noise_std = noise_std
        self.t = 0
        self.pos = None
        self.goal = None

    def reset(self, task: Point2DTask, init_pos: np.ndarray | None = None):
        self.t = 0
        self.goal = task.goal.astype(np.float32)
        if init_pos is None:
            self.pos = np.zeros(2, dtype=np.float32)
        else:
            self.pos = init_pos.astype(np.float32)
        return self.pos.copy()

    def step(self, action: np.ndarray):
        action = np.clip(action, -self.max_action, self.max_action).astype(np.float32)
        noise = np.random.randn(2).astype(np.float32) * self.noise_std
        self.pos = self.pos + action + noise
        self.t += 1

        # Dense reward: negative squared distance to the hidden goal.
        dist2 = float(np.sum((self.pos - self.goal) ** 2))
        reward = -dist2

        done = self.t >= self.horizon
        obs = self.pos.copy()
        info = {"dist2": dist2}
        return obs, reward, done, info

> **Implementation note:** the environment clips actions to `[-max_action, max_action]` as a
> safety net, but the *policy* enforces bounds via a tanh-squash transform, so the
> log-probabilities stay correct and the clip is (almost) never active.

In [ ]:
def sample_task(
    goal_range: float = 1.0,
    min_radius: float = 0.0,
    rng: np.random.RandomState | None = None,
) -> Point2DTask:
    """Sample a task goal from [-goal_range, goal_range]^2 with an optional min radius."""
    _rng = rng if rng is not None else np.random
    for _ in range(10000):
        goal = _rng.uniform(-goal_range, goal_range, size=(2,)).astype(np.float32)
        if min_radius <= 0.0 or float(np.linalg.norm(goal)) >= min_radius:
            return Point2DTask(goal=goal)
    return Point2DTask(goal=goal)  # fallback (shouldn't happen)

In [ ]:
# Quick sanity check
env = Point2DEnv(horizon=10, noise_std=0.0)
task = sample_task()
obs = env.reset(task)
for _ in range(3):
    obs, r, done, info = env.step(np.array([0.05, -0.02], dtype=np.float32))
obs, r, done, info

### Visualize a few sampled tasks


In [ ]:
def plot_tasks(tasks: List[Point2DTask], title: str = "Sampled goals"):
    goals = np.stack([t.goal for t in tasks], axis=0)
    plt.figure(figsize=(4, 4))
    plt.scatter(goals[:, 0], goals[:, 1], marker="x")
    plt.axhline(0, linewidth=1)
    plt.axvline(0, linewidth=1)
    plt.xlim(-1.2, 1.2)
    plt.ylim(-1.2, 1.2)
    plt.title(title)
    plt.xlabel("goal x")
    plt.ylabel("goal y")
    plt.gca().set_aspect("equal", adjustable="box")
    plt.show()


tasks = [sample_task() for _ in range(30)]
plot_tasks(tasks)

## Policy: a tiny Gaussian policy network


### A small but important detail: action bounds and log-prob correctness

In many continuous-control environments we need to **bound** actions (e.g. to
$[-a_{\max}, a_{\max}]$). A common mistake is:

1. sample $a \sim \mathcal{N}(\mu,\sigma)$
2. compute $\log \pi(a)$
3. **clip** the action before executing it in the environment

This makes the gradient estimator **biased**, because the action you executed is not the same random
variable you used in the log-prob term. To keep things **mathematically consistent**, we instead use
a **tanh-squashed Gaussian**:

- sample $u \sim \mathcal{N}(\mu,\sigma)$
- transform $a = a_{\text{scale}}\tanh(u)$
- use the _correct_ $\log \pi(a)$ including the Jacobian correction (handled automatically by
  PyTorch's `TransformedDistribution` + `TanhTransform`).


### Deterministic vs stochastic rollouts

Our policy is a Gaussian with a squashing nonlinearity (tanh). That means we can use it in two
modes:

- **Stochastic**: sample actions from the distribution. This is what we use for training, because
  policy gradients need exploration and log-probabilities.
- **Deterministic**: take the mean action (no sampling). This is great for _visualization_, because
  it produces clean, repeatable trajectories.

In the plots later:

- “before” trajectories are usually shown **stochastically** (often looks like a random walk without
  the goal).
- “after (det)” is shown **deterministically** (often a straight line toward the goal).


We use a diagonal Gaussian policy:

- Mean produced by an MLP: $\mu_\theta(s)$
- Log standard deviation is a learned parameter: $\log \sigma_\theta$

Actions are sampled as $a \sim \mathcal{N}(\mu_\theta(s), \sigma_\theta)$.


In [ ]:
class GaussianPolicy(nn.Module):
    """Gaussian policy with a tanh-squash transform.

    Why tanh-squash?
    - It bounds actions to (-action_scale, action_scale) in a differentiable way.
    - Crucially, it keeps the policy-gradient estimator *mathematically consistent*:
      the action executed in the environment is the same random variable whose log-probability we use.

    We'll implement π(a|s) by sampling u ~ Normal(μ(s), σ(s)), then:
        a = action_scale * tanh(u)

    PyTorch's TransformedDistribution + TanhTransform automatically accounts for the
    change-of-variables term in log_prob (the Jacobian correction).
    """

    def __init__(
        self,
        obs_dim: int,
        act_dim: int,
        hidden: int = 64,
        action_scale: float = 0.3,
        train_log_std: bool = False,
    ):
        super().__init__()
        self.fc1 = nn.Linear(obs_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.mu_head = nn.Linear(hidden, act_dim)

        # Initialize std to a modest value to reduce extreme actions early on.
        self.log_std = nn.Parameter(torch.ones(act_dim) * -1.0)
        # Freeze log_std by default (train_log_std=False) to keep exploration stable.
        self.log_std.requires_grad_(train_log_std)

        # Stabilizer: keep exploration within a narrow, safe range.
        # (Used in get_dist via clamp; helps prevent late-training variance explosions.)
        self.log_std_min = -3.0
        self.log_std_max = 1.0

        self.action_scale = float(action_scale)

    def get_param_dict(self) -> Dict[str, torch.Tensor]:
        """Return a parameter dict keyed by module/parameter name.

        This is convenient for functional MAML-style inner-loop updates, because the
        inner loop produces *new* parameter tensors (θ') from the loss without mutating
        the original nn.Module.
        """
        # TODO: Return a dict mapping parameter names to their tensors.
        # Hint: use self.named_parameters() which yields (name, Parameter) pairs.
        return None

    def _sanitize_stats(
        self, mu: torch.Tensor, log_std: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Clamp and (only if needed) de-NaN mu/log_std."""
        mu = mu.clamp(min=-2.0, max=2.0)
        log_std = log_std.clamp(min=self.log_std_min, max=self.log_std_max)

        # Note: clamp does *not* remove NaNs, so we keep a light guardrail.
        if not (torch.isfinite(mu).all() and torch.isfinite(log_std).all()):
            mu = torch.nan_to_num(mu, nan=0.0, posinf=0.0, neginf=0.0)
            log_std = torch.nan_to_num(log_std, nan=0.0, posinf=0.0, neginf=0.0)
        return mu, log_std

    def get_stats(
        self, obs: torch.Tensor, params: Dict[str, torch.Tensor] | None = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Return (mu, log_std) after clamping/sanitization.

        This is useful when we want statistics of the *pre-squash* Normal
        (e.g., an entropy term) without poking into distribution internals.
        """
        if params is None:
            mu, log_std = self.forward(obs)
        else:
            mu, log_std = self.forward_with_params(obs, params)
        return self._sanitize_stats(mu, log_std)

    def pre_squash_entropy(
        self, obs: torch.Tensor, params: Dict[str, torch.Tensor] | None = None
    ) -> torch.Tensor:
        """Entropy of the pre-squash Normal, summed across action dims.

        Returned shape: (batch,)
        """
        _, log_std = self.get_stats(obs, params=params)
        # Entropy(N(mu, sigma)) = sum_i (log(sigma_i) + 0.5*log(2*pi*e))
        const = 0.5 * math.log(2.0 * math.pi * math.e)
        return (log_std + const).sum(dim=-1)

    def forward(self, obs: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        h = torch.tanh(self.fc1(obs))
        h = torch.tanh(self.fc2(h))
        mu = self.mu_head(h)  # (batch, act_dim), unbounded
        log_std = self.log_std.expand_as(mu)
        return mu, log_std

    # Functional forward pass for MAML (explicit params dict).
    # Instead of using self.fcX.weight/bias (i.e., the nn.Module's own parameters),
    # we use the tensors passed in `params`. This lets the inner-loop update produce
    # adapted parameters θ' as *new tensors* while keeping the computation graph
    # so that higher-order derivatives can flow through them (needed for Full MAML).
    def forward_with_params(
        self, obs: torch.Tensor, params: Dict[str, torch.Tensor]
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # TODO: Re-implement the forward pass using F.linear and the weights/biases
        # from the `params` dict. The keys you need are:
        #   "fc1.weight", "fc1.bias", "fc2.weight", "fc2.bias",
        #   "mu_head.weight", "mu_head.bias", "log_std"
        # Follow the same architecture as `forward`: tanh(fc1) -> tanh(fc2) -> mu_head.
        # The log_std is just params["log_std"].expand_as(mu).
        h = None
        h = None
        mu = None
        log_std = None
        return mu, log_std

    def get_dist(self, obs: torch.Tensor, params: Dict[str, torch.Tensor] | None = None):
        """Return the squashed action distribution π(a|s)."""
        mu, log_std = self.get_stats(obs, params=params)
        std = torch.exp(log_std).clamp(min=1e-4)
        base = Normal(mu, std)

        # Squash with tanh, then scale to action_scale:
        #   a = action_scale * tanh(u)
        transforms = [
            TanhTransform(cache_size=1),
            AffineTransform(loc=0.0, scale=self.action_scale),
        ]
        dist = TransformedDistribution(base, transforms)
        dist = Independent(dist, 1)  # treat action dims jointly
        return dist


policy = GaussianPolicy(obs_dim=2, act_dim=2, hidden=64, action_scale=0.15).to(DEVICE)


In [ ]:
@contextmanager
def temp_seed(seed: int):
    """Temporarily set RNG seeds (py/random, numpy, torch) and restore after."""
    import random as _random
    import numpy as _np
    import torch as _torch

    py_state = _random.getstate()
    np_state = _np.random.get_state()
    torch_state = _torch.random.get_rng_state()
    cuda_state = _torch.cuda.get_rng_state_all() if _torch.cuda.is_available() else None

    _random.seed(seed)
    _np.random.seed(seed)
    _torch.manual_seed(seed)
    if _torch.cuda.is_available():
        _torch.cuda.manual_seed_all(seed)
    try:
        yield
    finally:
        _random.setstate(py_state)
        _np.random.set_state(np_state)
        _torch.random.set_rng_state(torch_state)
        if cuda_state is not None:
            _torch.cuda.set_rng_state_all(cuda_state)

## Rollouts and REINFORCE loss


We'll collect trajectories and compute a vanilla policy-gradient loss:

$$
L(\theta) \;=\; -\mathbb{E}\left[\sum_t \log \pi_\theta(a_t \mid s_t)\, G_t\right]
$$

where $G_t$ is the reward-to-go from time $t$.

> **Note:** $G_t$ is treated as a constant w.r.t. $\theta$ (no backprop through rewards), which
> matches the standard REINFORCE estimator.


In [ ]:
@torch.no_grad()
def rollout_episode(
    env: Point2DEnv,
    task: Point2DTask,
    policy: GaussianPolicy,
    params=None,
    deterministic_action: bool = False,
) -> dict:
    """Roll out one episode in the (numpy) environment.

    Always returns numpy arrays (no torch tensors) so the rollout function stays
    simple and side-effect free.

    For policy-gradient training, we recompute log-probabilities (and entropy)
    *later* from the saved (obs, act) sequences using the current `policy` and
    the provided `params`. This keeps gradients correct while avoiding a second
    "torch rollout" API.

    Returns a dict with:
      - obs:  (T+1, obs_dim) float32
      - act:  (T,   act_dim) float32
      - rew:  (T,)           float32
      - dist2:(T,)           float32 (diagnostic: squared distance to goal)
    """
    obs = env.reset(task)

    # Include the initial state so trajectories end at the final position.
    obs_list: list[np.ndarray] = [obs]
    act_list: list[np.ndarray] = []
    rew_list: list[float] = []
    dist2_list: list[float] = []

    for _ in range(env.horizon):
        obs_t = torch.as_tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)

        if deterministic_action:
            # Deterministic action = action_scale * tanh(mu)
            mu, _ = policy.get_stats(obs_t, params=params)
            action_t = torch.tanh(mu) * policy.action_scale
        else:
            dist = policy.get_dist(obs_t, params=params)
            action_t = dist.sample()

        # Step the numpy env.
        action = action_t.squeeze(0).cpu().numpy()
        obs2, rew, done, info = env.step(action)

        obs_list.append(obs2)
        act_list.append(action)
        rew_list.append(float(rew))
        dist2_list.append(float(info.get("dist2", np.nan)))

        obs = obs2
        if done:
            break

    return {
        "obs": np.array(obs_list, dtype=np.float32),
        "act": np.array(act_list, dtype=np.float32),
        "rew": np.array(rew_list, dtype=np.float32),
        "dist2": np.array(dist2_list, dtype=np.float32),
    }

### A tiny utility to visualize a trajectory


In [ ]:
def plot_trajectory(ep: dict, task: Point2DTask, title: str = ""):
    traj = ep["obs"]
    goal = task.goal
    plt.figure(figsize=(4, 4))
    plt.plot(traj[:, 0], traj[:, 1], marker="o", markersize=3, linewidth=1)
    plt.scatter([0.0], [0.0], marker="s", label="start")
    plt.scatter([goal[0]], [goal[1]], marker="x", s=80, label="goal")
    plt.axhline(0, linewidth=1)
    plt.axvline(0, linewidth=1)
    plt.xlim(-1.4, 1.4)
    plt.ylim(-1.4, 1.4)
    plt.gca().set_aspect("equal", adjustable="box")
    plt.title(title)
    plt.legend()
    plt.show()


# Random untrained trajectory
env = Point2DEnv(horizon=25, noise_std=0.0)
task = sample_task()
ep = rollout_episode(env, task, policy)
plot_trajectory(ep, task, title="Untrained policy (random-ish)")

## MAML vs FOMAML (where the second derivative appears)


MAML learns an initialization $\theta$ such that a small number of gradient steps yields good
task-specific parameters.

**Inner-loop adaptation** (on a support set):

$$
\theta' = \theta - \alpha \nabla_\theta L_{\text{support}}(\theta)
$$

**Outer-loop meta-objective** (evaluate after adaptation on a query set):

$$
\min_\theta\; L_{\text{query}}(\theta')
$$

### Full MAML gradient

By the chain rule:

$$
\nabla_\theta L_{\text{query}}(\theta') =
\underbrace{\nabla_{\theta'} L_{\text{query}}(\theta')}_{\text{gradient at adapted params}}
\cdot
\underbrace{\frac{\partial \theta'}{\partial \theta}}_{\text{depends on Hessian}}
$$

Now:

$$
\frac{\partial \theta'}{\partial \theta}
= I - \alpha \nabla^2_\theta L_{\text{support}}(\theta)
$$

That Hessian term is where the **second-order derivative** enters.

### FOMAML approximation

FOMAML drops the Hessian term and approximates:

$$
\frac{\partial \theta'}{\partial \theta} \approx I
$$

So the meta-gradient becomes:

$$
\nabla_\theta L_{\text{query}}(\theta') \approx \nabla_{\theta'} L_{\text{query}}(\theta')
$$

### In PyTorch

- **Full MAML**: compute the inner-loop gradient with `create_graph=True` so the computation graph
  retains the dependency needed for second-order derivatives.
- **FOMAML**: use `create_graph=False` in the inner-loop gradient; gradients are treated as
  constants, so the Hessian term is ignored.

**Important:** with policy gradients, the usual REINFORCE surrogate loss has the right _first-order_
gradient, but its **higher-order** derivatives are biased unless you use a score-function estimator
designed for higher-order differentiation (e.g. **DiCE**). In this notebook, when
`second_order=True` we use a DiCE-style inner loss for the **support** step, and we also use a
smaller outer-loop learning rate to keep training stable.


## MAML implementation


In [ ]:
def adapt_params(
    params: Dict[str, torch.Tensor],
    loss: torch.Tensor,
    step_size: float,
    second_order: bool,
    max_grad_norm: float | None = None,
) -> Dict[str, torch.Tensor]:
    """Take one gradient step on a dict of parameters.

    This is the MAML **inner-loop update**:
        θ' = θ - α ∇_θ L_support(θ)

    The critical detail for Full MAML is that θ' must remain a *function* of θ in the
    computation graph, so that ∇_θ L_query(θ') can be computed through it later.
    That's what `second_order=True` enables.

    Supports frozen parameters (requires_grad=False), which is useful for keeping
    exploration (log_std) fixed in this educational demo.
    """
    trainable_items = [(k, v) for k, v in params.items() if v.requires_grad]
    if len(trainable_items) == 0:
        return dict(params)

    keys, tensors = zip(*trainable_items)

    # TODO: Compute gradients of `loss` with respect to `tensors` using torch.autograd.grad.
    # For Full MAML (second_order=True) we need the computation graph to be retained so
    # that higher-order derivatives can flow through this step. For FOMAML (False),
    # we treat the gradients as constants.
    # Hint: torch.autograd.grad(loss, list(tensors), create_graph=..., retain_graph=..., allow_unused=False).
    # Both `create_graph` and `retain_graph` should equal `second_order`.
    grads = None

    # Fail fast if gradients become non-finite (easier to debug than silently zeroing them).
    for k, g in zip(keys, grads):
        if not torch.isfinite(g).all():
            raise RuntimeError(
                f"Non-finite gradient for {k}. Try smaller step_size or enable grad clipping."
            )

    if max_grad_norm is not None:
        total_norm = torch.sqrt(sum(g.pow(2).sum() for g in grads))
        clip_coef = max_grad_norm / (total_norm + 1e-6)
        clip_coef = torch.clamp(clip_coef, max=1.0)
        grads = [g * clip_coef for g in grads]

    # TODO: Build the updated parameter dict. Start from a copy of `params` (so that
    # non-trainable entries are preserved unchanged), then for each trainable
    # (key, tensor, grad) write new_params[key] = tensor - step_size * grad.
    new_params = dict(params)
    # ...
    return new_params


### Differentiable rollouts for MAML


To compute gradients, we need log-probabilities as tensors (not detached scalars). We'll implement a
rollout function that returns log-probs and rewards as torch tensors.

The environment itself is still a small NumPy loop (that's fine), because REINFORCE does not
backprop through dynamics or reward.


### Intermezzo: the DiCE estimator and the magic-box operator

The REINFORCE loss below uses the classic score-function trick: multiply the reward-to-go by the
log-probability and _detach_ the reward term so gradients only flow through $\log \pi$.

An alternative formulation — the **DiCE** (Differentiate through the Cost and Effects) estimator —
avoids the explicit `detach()` by wrapping the cumulative log-probability in a **magic-box**
operator:

$$
\Box(x) = \exp\bigl(x - \text{stop\_grad}(x)\bigr)
$$

Its forward value is always **1** (so it doesn't change the loss magnitude), but its gradient equals
$\nabla x$ — meaning gradients flow through it correctly. DiCE uses per-timestep discounted rewards
(not reward-to-go) and the cumulative sum of log-probs, producing a gradient estimator that is
mathematically equivalent to REINFORCE but designed for **higher-order differentiation** — exactly
what full MAML needs.

In this notebook:

- **FOMAML** (`second_order=False`): we use the standard REINFORCE formulation (`use_dice=False`)
  for the inner loop, since no second derivative is needed.
- **Full MAML** (`second_order=True`): we use the DiCE estimator (`use_dice=True`) for the inner
  (support) loss, so that higher-order gradients through the inner update are unbiased.

The outer (query) loss always uses standard REINFORCE, since only first-order gradients flow through
it.

In [ ]:
def magic_box(x: torch.Tensor) -> torch.Tensor:
    """DiCE magic-box operator.

    Forward value is 1, but gradients flow through `x`:
        ∇ □(x) = ∇ x

    This lets us write losses whose forward value equals the scalar reward-weighted
    sum, while the backward pass produces the correct policy-gradient estimator
    (including higher-order derivatives) without any explicit detach().
    """
    # TODO: Implement the magic-box operator: exp(x - stop_gradient(x)).
    # Hint: use x.detach() for the stop-gradient version of x.
    return None


# Quick demo: forward value is 1, but gradient is the gradient of x
x = torch.tensor(2.0, requires_grad=True)
y = magic_box(x) * x**2  # forward: 1 * 4 = 4
y.backward()
print(f"magic_box forward: {magic_box(x).item():.4f} (always 1)")
print(f"grad of (magic_box(x) * x^2) w.r.t. x: {x.grad.item():.4f}")
print(f"  (= d/dx [x^2] * 1 + x^2 * d/dx [magic_box(x)] = 2x + x^2 = {2*2 + 4:.1f})")


In [ ]:
def reward_to_go(rewards: torch.Tensor, gamma: float = 1.0) -> torch.Tensor:
    # rewards: (T,)
    T = rewards.shape[0]
    rtg = torch.zeros_like(rewards)
    # TODO: Compute the discounted reward-to-go G_t = r_t + γ r_{t+1} + γ² r_{t+2} + ...
    # Walk backwards through the time dimension so each G_t can build on G_{t+1}.
    # Hint: iterate with `for t in reversed(range(T))` and keep a running accumulator.
    # ...
    return rtg


def reinforce_loss_from_episodes_torch(
    episodes: List[dict],
    policy: GaussianPolicy,
    params: Dict[str, torch.Tensor] | None,
    gamma: float = 1.0,
    return_scale: float = 0.1,
    use_dice: bool = False,
    entropy_coef: float = 0.0,
) -> torch.Tensor:
    """Compute a REINFORCE-style loss from numpy rollouts.

    Episodes are expected to come from `rollout_episode(...)`, i.e. each dict has:
      - "obs": (T+1, obs_dim) np.ndarray
      - "act": (T,   act_dim) np.ndarray
      - "rew": (T,)          np.ndarray

    We recompute log-probabilities (and an optional entropy term) under the policy
    using the saved (obs, act) sequences. This keeps gradients correct w.r.t.
    `params`, while allowing the rollout function itself to stay numpy-only.
    """
    if len(episodes) == 0:
        raise ValueError("episodes must be non-empty")

    processed: list[dict[str, torch.Tensor | None]] = []
    all_vals: list[torch.Tensor] = []

    # Small clamp to keep inverse-tanh stable in TransformedDistribution.log_prob.
    scale = float(policy.action_scale)
    eps = 1e-6

    for ep in episodes:
        obs_np: np.ndarray = ep["obs"]
        act_np: np.ndarray = ep["act"]
        rew_np: np.ndarray = ep["rew"]

        # Policy is evaluated at the state where the action was chosen: obs[t] -> act[t]
        obs_t = torch.as_tensor(obs_np[:-1], dtype=torch.float32, device=DEVICE)  # (T,obs)
        act_t = torch.as_tensor(act_np, dtype=torch.float32, device=DEVICE)  # (T,act)
        rewards = torch.as_tensor(rew_np, dtype=torch.float32, device=DEVICE)  # (T,)

        act_t = act_t.clamp(min=-scale + eps, max=scale - eps)

        # TODO: Compute log-probabilities of the taken actions under the current policy,
        # using the *functional* forward pass (i.e., pass `params=params` through).
        # Hint: policy.get_dist(obs_t, params=params).log_prob(act_t) gives shape (T,).
        dist = None
        logp = None  # shape (T,)

        ent: torch.Tensor | None = None
        if entropy_coef:
            ent = policy.pre_squash_entropy(obs_t, params=params)  # (T,)

        # TODO: Compute the per-timestep scalar values used by the loss.
        # - Standard REINFORCE (use_dice=False): use the reward-to-go G_t.
        # - DiCE (use_dice=True): use per-timestep *discounted* rewards γ^t * r_t
        #   (NOT reward-to-go — DiCE multiplies each r_t by the cumulative logp up to t).
        if use_dice:
            # Hint: build discounts = gamma ** arange(T), then val = rewards * discounts.
            T = rewards.shape[0]
            val = None
        else:
            # Hint: val = reward_to_go(rewards, gamma=gamma).
            val = None

        processed.append({"val": val, "logp": logp, "ent": ent})
        all_vals.append(val)

    # Batch-wide baseline/normalization.
    flat_vals = torch.cat(all_vals)
    mean = flat_vals.mean()
    std = flat_vals.std(unbiased=False).clamp_min(1e-8)

    losses: list[torch.Tensor] = []
    for item in processed:
        val = item["val"]
        logp = item["logp"]

        adv = ((val - mean) / std) * return_scale

        # TODO: Compute the per-episode loss.
        # - Standard PG: loss = -mean(logp * adv.detach())
        #   (we detach `adv` so gradients only flow through logp, matching REINFORCE).
        # - DiCE: loss = -mean(magic_box(cumsum(logp, dim=0)) * adv)
        #   (cumulative sum of logp wrapped in the magic-box operator).
        if use_dice:
            loss = None
        else:
            loss = None

        ent = item["ent"]
        if entropy_coef and ent is not None:
            loss = loss - entropy_coef * ent.mean()

        losses.append(loss)

    return torch.stack(losses).mean()


### Baseline: can a random policy adapt?

Before we start meta-training, let's check whether a freshly initialized (untrained) policy can
adapt to a new task using just a few gradient steps. If vanilla fine-tuning already works, we
wouldn't need meta-learning at all.

In [ ]:
# Baseline: attempt to adapt a *random* (non-meta-trained) policy.
# This is a nice warm-up: it reuses `adapt_params` and `reinforce_loss_from_episodes_torch`
# to perform *just* the inner loop of MAML — without any meta-training. If meta-learning
# is doing anything, a random init + K gradient steps should be clearly worse than a
# meta-trained init + K gradient steps.
def baseline_adapt_test(
    seed: int = 42, inner_steps: int = 3, inner_lr: float = 0.05, support_episodes: int = 4
):
    init_random(seed=seed)
    random_policy = GaussianPolicy(obs_dim=2, act_dim=2, hidden=64, action_scale=0.15).to(DEVICE)
    task = sample_task(goal_range=1.0, min_radius=0.3)
    env = Point2DEnv(horizon=25, max_action=0.15, noise_std=0.0)

    theta = random_policy.get_param_dict()

    # Before adaptation
    ep_before = rollout_episode(env, task, random_policy, params=theta, deterministic_action=True)
    ret_before = float(ep_before["rew"].sum())

    # TODO: Implement the adaptation loop — this is the MAML *inner loop* in isolation.
    # For each of `inner_steps` steps:
    #   1. Collect `support_episodes` rollouts using the current `theta_k`
    #      (stochastic — we need exploration to get informative gradients).
    #      Hint: rollout_episode(env, task, random_policy, params=theta_k).
    #   2. Compute a REINFORCE support loss via reinforce_loss_from_episodes_torch(...)
    #      with gamma=1.0, return_scale=0.5, use_dice=False.
    #   3. Update theta_k by calling adapt_params(theta_k, loss, step_size=inner_lr,
    #      second_order=False).
    theta_k = theta
    for _ in range(inner_steps):
        pass  # TODO

    # After adaptation
    ep_after = rollout_episode(env, task, random_policy, params=theta_k, deterministic_action=True)
    ret_after = float(ep_after["rew"].sum())

    print(
        f"Random policy — before: {ret_before:.2f}, after {inner_steps} steps: {ret_after:.2f}, "
        f"gain: {ret_after - ret_before:.2f}"
    )
    plot_trajectory(ep_before, task, title="Random policy — before adaptation")
    plot_trajectory(ep_after, task, title=f"Random policy — after {inner_steps} adaptation steps")


baseline_adapt_test()


As expected, the random policy barely improves — a few gradient steps are not enough to learn a good
policy from scratch. Meta-learning will change this: by meta-training the initialization, those same
few steps will produce dramatic improvement.


## Training configuration

A few notes on the hyperparameters below:

- **`tasks_per_batch`**: how many tasks we sample per meta-update. More tasks = lower variance
  meta-gradient, but slower iterations.
- **`inner_steps` / `inner_lr`**: number and size of adaptation gradient steps. Too many or too
  large can overshoot; too few won't adapt.
- **`support_episodes` / `query_episodes`**: rollouts collected for the inner and outer loops
  respectively. More episodes reduce variance but cost more compute.
- **`return_scale`**: after standardizing returns across the meta-batch, we scale them by this
  factor. This controls the effective magnitude of the policy gradient and helps prevent updates
  that are too aggressive.
- **`entropy_coef`**: small exploration bonus added to the support loss to keep gradients
  informative (only used in the inner loop).
- **`goal_min_radius`**: tasks with goals too close to the origin are trivial (the agent starts
  there), so we enforce a minimum distance.
- **`inner_max_grad_norm` / `max_grad_norm`**: gradient clipping to prevent instability.


In [ ]:
@dataclass
class MetaConfig:
    # Keep this fast enough to iterate, but stable enough to see adaptation.
    meta_iters: int = 600
    tasks_per_batch: int = 10

    # Inner loop (adaptation)
    inner_steps: int = 3
    inner_lr: float = 0.05
    support_episodes: int = 4

    # Outer loop
    query_episodes: int = 4
    meta_lr: float = 3e-4
    meta_lr_second_order: float = 2e-4

    gamma: float = 1.0
    # Note: this notebook always standardizes returns across the meta-batch (see REINFORCE loss).
    return_scale: float = 0.5

    # Exploration bonus coefficient (support loss only; helps keep gradients informative)
    entropy_coef: float = 0.02

    # Stabilizer: softly keep log_std near its initialization (kept off by default now).
    std_reg_coef: float = 0.0
    std_reg_target: float = -0.7

    # Sample tasks away from the origin so adaptation is meaningful (pedagogical).
    goal_min_radius: float = 0.3

    # Switch: full MAML vs FO approximation
    second_order: bool = False  # False=FOMAML (faster), True=Full MAML (slower)

    inner_max_grad_norm: float | None = 5.0
    max_grad_norm: float | None = 5.0


cfg = MetaConfig()
cfg

## Meta-training loop


In [ ]:
def _val_score_after_adaptation(
    policy: GaussianPolicy,
    tasks: List[Point2DTask],
    cfg: MetaConfig,
    env_kwargs: dict,
    goal_range: float,
    eval_seed: int = 123,
    eval_deterministic: bool = True,
    n_rollouts: int = 1,
) -> Tuple[float, float, float]:
    """Return (mean_before, mean_after, mean_gain) on a fixed task set.

    This is used for checkpointing / early stopping. For stability and speed in an
    educational notebook, we default to deterministic actions and a small number of rollouts.
    """
    # Deterministic seeding for evaluation reproducibility
    rng = np.random.RandomState(eval_seed)

    r0s: list[float] = []
    rks: list[float] = []

    for task in tasks:
        # Make sure any per-episode randomness is repeatable across calls
        np.random.seed(rng.randint(0, 2**31 - 1))
        torch.manual_seed(rng.randint(0, 2**31 - 1))

        env = Point2DEnv(**env_kwargs)

        theta = policy.get_param_dict()

        # BEFORE adaptation
        for _ in range(max(1, n_rollouts)):
            ep0 = rollout_episode(
                env, task, policy, params=theta, deterministic_action=eval_deterministic
            )
            r0s.append(float(ep0["rew"].sum()))

        # Adapt for K steps on support episodes (stochastic, no entropy bonus)
        theta_k = theta
        for _k in range(cfg.inner_steps):
            support_eps = [
                rollout_episode(env, task, policy, params=theta_k)
                for _ in range(cfg.support_episodes)
            ]
            loss = reinforce_loss_from_episodes_torch(
                support_eps,
                policy,
                theta_k,
                gamma=cfg.gamma,
                return_scale=cfg.return_scale,
                use_dice=False,
                entropy_coef=0.0,
            )
            theta_k = adapt_params(
                theta_k,
                loss,
                step_size=cfg.inner_lr,
                second_order=False,
                max_grad_norm=cfg.inner_max_grad_norm,
            )

        # AFTER adaptation
        for _ in range(max(1, n_rollouts)):
            epk = rollout_episode(
                env, task, policy, params=theta_k, deterministic_action=eval_deterministic
            )
            rks.append(float(epk["rew"].sum()))

    before = float(np.mean(r0s)) if len(r0s) else -float("inf")
    after = float(np.mean(rks)) if len(rks) else -float("inf")
    gain = after - before
    return before, after, gain


def meta_train(
    policy: GaussianPolicy,
    cfg: MetaConfig,
    env_kwargs: dict | None = None,
    goal_range: float = 1.0,
    *,
    eval_every: int = 50,
    n_val_tasks: int = 20,
    val_rollouts: int = 1,
    val_seed: int = 123,
    eval_deterministic: bool = True,
    early_stop_patience: int = 4,
    early_stop_min_delta: float = 1e-3,
):
    """Meta-train with an explicit best-checkpoint on a fixed validation set.

    This makes the notebook *reliably* show adaptation, even when RL training is non-monotonic.
    """
    if env_kwargs is None:
        env_kwargs = dict(horizon=25, max_action=0.15, noise_std=0.00)

    meta_lr = cfg.meta_lr_second_order if cfg.second_order else cfg.meta_lr
    optimizer = torch.optim.Adam(policy.parameters(), lr=meta_lr)

    # Fixed validation task set (doesn't touch global RNG)
    rng = np.random.RandomState(val_seed)
    val_tasks = [
        sample_task(goal_range=goal_range, min_radius=cfg.goal_min_radius, rng=rng)
        for _ in range(n_val_tasks)
    ]

    history = {
        "meta_loss": [],
        "meta_loss_total": [],
        "query_return_mean": [],
        "query_return_std": [],
        # Validation metrics are recorded at evaluation checkpoints as (iter, value)
        "val_return_before": [],
        "val_return_after": [],
        "val_return_gain": [],
        # Best checkpoint selection uses post-adaptation return (`val_after`) by default.
        # `val_gain` is still logged for pedagogy, but it can peak early as `val_before` improves.
        "best_iter": None,
        "best_val_gain": -float("inf"),
        "best_val_after": -float("inf"),
        "best_val_before": None,
        "best_val_gain_at_best": None,
    }

    best_state = copy.deepcopy(policy.state_dict())
    evals_since_improve = 0

    for it in range(cfg.meta_iters):
        optimizer.zero_grad()

        # Sample a batch of tasks
        rng_train = np.random.RandomState(np.random.randint(0, 2**31 - 1))
        tasks = [
            sample_task(goal_range=goal_range, min_radius=cfg.goal_min_radius, rng=rng_train)
            for _ in range(cfg.tasks_per_batch)
        ]

        task_query_returns = []
        task_query_losses = []

        for task in tasks:
            env = Point2DEnv(**env_kwargs)

            # Current parameters theta (the meta-parameters we want to improve)
            theta = policy.get_param_dict()

            # TODO: ----- INNER LOOP: adapt on support episodes -----
            # Starting from theta_i = theta, take `cfg.inner_steps` inner-loop updates.
            # At each step:
            #   1. Collect `cfg.support_episodes` rollouts with the *current* theta_i
            #      using rollout_episode(env, task, policy, params=theta_i).
            #   2. Compute a support loss with reinforce_loss_from_episodes_torch(...),
            #      passing:
            #        - gamma=cfg.gamma, return_scale=cfg.return_scale
            #        - use_dice=cfg.second_order  (so Full MAML uses the DiCE estimator
            #          which is designed for higher-order differentiation)
            #        - entropy_coef=getattr(cfg, "entropy_coef", 0.0)
            #   3. Update theta_i with adapt_params(...), passing:
            #        - step_size=cfg.inner_lr
            #        - second_order=cfg.second_order  (controls whether we retain the
            #          graph for the meta-gradient through this update)
            #        - max_grad_norm=cfg.inner_max_grad_norm
            theta_i = theta
            for _ in range(cfg.inner_steps):
                pass  # TODO

            # TODO: ----- OUTER LOOP: evaluate on query episodes with adapted params θ' -----
            # 1. Collect `cfg.query_episodes` rollouts using the adapted parameters theta_i.
            # 2. Compute the query loss with reinforce_loss_from_episodes_torch(...),
            #    passing use_dice=False (the outer loss only needs first-order gradients,
            #    which flow from θ' back to θ through the inner update's computation graph).
            # 3. Append the query loss to `task_query_losses`.
            query_eps = None
            query_loss = None
            # task_query_losses.append(query_loss)

            # For reporting: undiscounted return per episode (this already works once
            # `query_eps` is populated above).
            with torch.no_grad():
                if query_eps is not None:
                    rets = [float(ep["rew"].sum().item()) for ep in query_eps]
                    task_query_returns.append(np.mean(rets))

        meta_loss_raw = torch.stack(task_query_losses).mean()

        # Stabilizer: softly regularize exploration scale (log_std) to stay near initialization.
        # This reduces late-training drift/collapse in high-variance RL meta-gradients.
        if getattr(cfg, "std_reg_coef", 0.0) and cfg.std_reg_coef > 0:
            theta0 = policy.get_param_dict()
            std_reg = ((theta0["log_std"] - cfg.std_reg_target) ** 2).mean()
            meta_loss = meta_loss_raw + cfg.std_reg_coef * std_reg
        else:
            meta_loss = meta_loss_raw

        meta_loss.backward()
        if cfg.max_grad_norm is not None:
            torch.nn.utils.clip_grad_norm_(policy.parameters(), cfg.max_grad_norm)
        optimizer.step()

        history["meta_loss"].append(float(meta_loss_raw.item()))
        history["meta_loss_total"].append(float(meta_loss.item()))
        history["query_return_mean"].append(float(np.mean(task_query_returns)))
        history["query_return_std"].append(float(np.std(task_query_returns)))

        # Validation + checkpoint
        do_eval = (it == 0) or ((it + 1) % eval_every == 0) or (it + 1 == cfg.meta_iters)
        if do_eval:
            vb, va, vg = _val_score_after_adaptation(
                policy,
                val_tasks,
                cfg,
                env_kwargs=env_kwargs,
                goal_range=goal_range,
                eval_seed=val_seed,
                eval_deterministic=eval_deterministic,
                n_rollouts=val_rollouts,
            )
            history["val_return_before"].append((it + 1, vb))
            history["val_return_after"].append((it + 1, va))
            history["val_return_gain"].append((it + 1, vg))

            # Checkpoint selection: improve post-adaptation return (higher is better)
            if va > history["best_val_after"] + early_stop_min_delta:
                history["best_val_after"] = va
                history["best_iter"] = it + 1
                history["best_val_before"] = vb
                history["best_val_gain_at_best"] = vg
                best_state = copy.deepcopy(policy.state_dict())
                evals_since_improve = 0
            else:
                evals_since_improve += 1

            # Track best gain for logging only (gain can peak early as `val_before` improves)
            if vg > history["best_val_gain"]:
                history["best_val_gain"] = vg

            if early_stop_patience is not None and evals_since_improve >= early_stop_patience:
                history["stopped_early"] = True
                history["stop_iter"] = it + 1
                print(
                    f"Early stopping: no val_after improvement for {early_stop_patience} evals. Stopping at iter {it+1}."
                )
                break  # early stop

        # Logging
        if (it + 1) % 25 == 0 or it == 0:
            so = "Full MAML" if cfg.second_order else "FOMAML"
            msg = (
                f"[{so}] iter {it+1:4d}/{cfg.meta_iters} | meta_loss {meta_loss_raw.item():.4f}"
                f" | train_query_ret {history['query_return_mean'][-1]:.2f} ± {history['query_return_std'][-1]:.2f}"
            )
            if do_eval:
                _, last_vb = history["val_return_before"][-1]
                _, last_va = history["val_return_after"][-1]
                _, last_vg = history["val_return_gain"][-1]

                best_after = history["best_val_after"]
                best_it = history["best_iter"]
                best_gain = history["best_val_gain"]
                gain_at_best = history.get("best_val_gain_at_best", None)

                if gain_at_best is None:
                    best_str = (
                        f"(best after {best_after:.2f} @ {best_it}; best gain {best_gain:.2f})"
                    )
                else:
                    best_str = (
                        f"(best after {best_after:.2f} @ {best_it}; gain@best {gain_at_best:.2f};"
                        f" best gain {best_gain:.2f})"
                    )

                msg += f" | val_before {last_vb:.2f} | val_after {last_va:.2f} | val_gain {last_vg:.2f} {best_str}"
            print(msg)

    # Restore best checkpoint
    policy.load_state_dict(best_state)
    return history


# Train (with best-checkpoint selection on a fixed validation set)
init_random(seed=0)
policy = GaussianPolicy(obs_dim=2, act_dim=2, hidden=64, action_scale=0.15).to(DEVICE)

history = meta_train(
    policy,
    cfg,
    goal_range=0.7,
    eval_every=50,
    n_val_tasks=30,
    val_rollouts=1,
    val_seed=123,
    eval_deterministic=True,
    early_stop_patience=3,
    early_stop_min_delta=5e-3,
)
print(
    f"Restored best checkpoint from iter {history['best_iter']} (best val_gain={history['best_val_gain']:.2f}, val_after={history['best_val_after']:.2f})"
)

### Plot training curve

**What to look for:** the meta-loss should trend downward, but expect non-monotonic behavior — this
is normal in RL. The validation gain (if shown) matters more than the raw loss value.


In [ ]:
def plot_training(history, title_prefix="", smooth_window: int = 25):
    x = np.arange(1, len(history["meta_loss"]) + 1)

    def smooth(y):
        y = np.asarray(y, dtype=np.float32)
        if smooth_window <= 1 or len(y) < smooth_window:
            return x, y
        w = np.ones(smooth_window, dtype=np.float32) / float(smooth_window)
        ys = np.convolve(y, w, mode="valid")
        xs = np.arange(smooth_window, len(y) + 1)
        return xs, ys

    plt.figure(figsize=(6, 3))
    plt.plot(x, history["meta_loss"], alpha=0.25, label="raw")
    xs, ys = smooth(history["meta_loss"])
    plt.plot(xs, ys, label=f"MA({smooth_window})")
    plt.title(f"{title_prefix}Meta loss")
    plt.xlabel("meta-iteration")
    plt.ylabel("loss")
    plt.legend()
    plt.show()

    plt.figure(figsize=(6, 3))
    plt.plot(x, history["query_return_mean"], alpha=0.25, label="raw")
    xs, ys = smooth(history["query_return_mean"])
    plt.plot(xs, ys, label=f"MA({smooth_window})")

    # Overlay validation checkpoints (deterministic eval), if present
    if "val_return_after" in history and len(history["val_return_after"]) > 0:
        its, vals = zip(*history["val_return_after"])
        plt.scatter(its, vals, marker="x", label="val_after", alpha=1.0)
    if "val_return_before" in history and len(history["val_return_before"]) > 0:
        itsb, valsb = zip(*history["val_return_before"])
        plt.scatter(itsb, valsb, marker="o", label="val_before", alpha=1.0)
    plt.title(f"{title_prefix}Query return (after adaptation)")
    plt.xlabel("meta-iteration")
    plt.ylabel("average return")
    plt.legend()
    plt.show()

    # Plot validation gain (after-before), which is the main "adaptation works" signal.
    if "val_return_gain" in history and len(history["val_return_gain"]) > 0:
        itsg, valg = zip(*history["val_return_gain"])
        plt.figure(figsize=(6, 3))
        plt.plot(itsg, valg, marker="o", linewidth=1, label="val_gain = after - before")
        plt.axhline(0.0, linewidth=1)
        plt.title(f"{title_prefix}Validation gain from adaptation")
        plt.xlabel("meta-iteration")
        plt.ylabel("gain in return")
        plt.legend()
        plt.show()


plot_training(history, title_prefix=("Full MAML - " if cfg.second_order else "FOMAML - "))

## Evaluate fast adaptation on unseen tasks

**What to look for:** the mean return should jump noticeably after just 1-3 inner gradient steps,
confirming that the meta-learned initialization is easy to fine-tune.


## What does “adaptation” mean here?

For a single task (hidden goal), “adaptation” means taking a few gradient steps on **support**
experience and getting new parameters \(\theta'\) that perform better on **query** experience from
the same task.

We measure adaptation with:

- **Return before**: average episodic return using the _initial_ parameters \(\theta\).
- **Return after**: average episodic return using the _adapted_ parameters \(\theta'\).
- **Gain**: \(\text{return_after} - \text{return_before}\).

In this notebook we use deterministic evaluation for these metrics (mean-action rollouts) to reduce
noise.


We will sample new tasks (new hidden goals), then compare:

- **Before adaptation:** rollout with $\theta$
- **After k adaptation steps:** rollout with $\theta^{(k)}$

We expect meta-trained policies to improve quickly after just a few gradient steps.

A few notes to interpret the plots in this notebook:

- The reward is `reward = -||pos - goal||^2`, so **returns are negative** and the best possible
  return is **0** (if you reach the goal and stay there).
- With `goal_range=1.0` and `horizon=25`, a policy that stays near the origin gets about
  `E[return] ≈ -25 * E[||goal||^2] = -25 * (2/3) ≈ -16.7`. So **"random-walk near the origin"** is a
  common _pre-adaptation_ behavior.
- **After adaptation**, a good trajectory should move **toward the sampled goal** and finish much
  closer to it (returns move toward 0).


In [ ]:
def evaluate_adaptation_curve(
    policy: GaussianPolicy,
    *,
    n_tasks: int = 20,
    goal_range: float = 1.0,
    k_steps: int = 3,
    inner_lr: float = 0.02,
    support_episodes: int = 5,
    env_kwargs=None,
    gamma: float = 1.0,
    return_scale: float = 0.1,
    inner_max_grad_norm: float | None = 5.0,
    eval_seed: int = 999,
    eval_rollouts: int = 5,
    eval_deterministic: bool = True
) -> dict:
    """Returns a matrix of deterministic returns for k=0..k_steps (shape: [k+1, n_tasks])."""
    if env_kwargs is None:
        env_kwargs = dict(horizon=25, max_action=0.15, noise_std=0.00)

    rng = np.random.RandomState(eval_seed)
    tasks = [
        Point2DTask(goal=rng.uniform(-goal_range, goal_range, size=(2,)).astype(np.float32))
        for _ in range(n_tasks)
    ]

    returns_det = np.zeros((k_steps + 1, n_tasks), dtype=np.float32)
    returns_stoch = np.zeros((k_steps + 1, n_tasks), dtype=np.float32)

    # Keep evaluation reproducible while not perturbing training RNG
    with temp_seed(eval_seed):
        for ti, task in enumerate(tasks):
            env = Point2DEnv(**env_kwargs)
            theta = policy.get_param_dict()

            # k = 0 (before adaptation)
            ep0 = rollout_episode(
                env, task, policy, params=theta, deterministic_action=eval_deterministic
            )
            returns_det[0, ti] = float(ep0["rew"].sum())
            rs0 = []
            for _ in range(eval_rollouts):
                ep0s = rollout_episode(env, task, policy, params=theta, deterministic_action=False)
                rs0.append(float(ep0s["rew"].sum()))
            returns_stoch[0, ti] = float(np.mean(rs0))

            theta_k = theta
            for k in range(1, k_steps + 1):
                support_eps = [
                    rollout_episode(env, task, policy, params=theta_k)
                    for _ in range(support_episodes)
                ]
                loss = reinforce_loss_from_episodes_torch(
                    support_eps, policy, theta_k, gamma=gamma, return_scale=return_scale
                )
                theta_k = adapt_params(
                    theta_k,
                    loss,
                    step_size=inner_lr,
                    second_order=False,
                    max_grad_norm=inner_max_grad_norm,
                )

                epk = rollout_episode(
                    env, task, policy, params=theta_k, deterministic_action=eval_deterministic
                )
                returns_det[k, ti] = float(epk["rew"].sum())
                rsk = []
                for _ in range(eval_rollouts):
                    epks = rollout_episode(
                        env, task, policy, params=theta_k, deterministic_action=False
                    )
                    rsk.append(float(epks["rew"].sum()))
                returns_stoch[k, ti] = float(np.mean(rsk))

    return {"tasks": tasks, "returns_det": returns_det, "returns_stoch": returns_stoch}


def plot_adaptation_curve(curve: dict, title: str = "Adaptation curve"):
    rets_det = curve["returns_det"]
    rets_stoch = curve["returns_stoch"]
    ks = np.arange(rets_det.shape[0])
    mean_det = rets_det.mean(axis=1)
    std_det = rets_det.std(axis=1)
    mean_st = rets_stoch.mean(axis=1)
    std_st = rets_stoch.std(axis=1)

    plt.figure(figsize=(6, 3))
    plt.errorbar(ks, mean_det, yerr=std_det, capsize=3, label="deterministic")
    plt.errorbar(ks, mean_st, yerr=std_st, capsize=3, label="stochastic")
    plt.xticks(ks)
    plt.xlabel("Number of inner updates")
    plt.ylabel("Return (higher is better)")
    plt.title(title)
    plt.legend()
    plt.show()


curve = evaluate_adaptation_curve(
    policy,
    n_tasks=20,
    goal_range=1.0,
    k_steps=cfg.inner_steps,
    inner_lr=cfg.inner_lr,
    support_episodes=2,
    gamma=cfg.gamma,
    return_scale=cfg.return_scale,
    inner_max_grad_norm=cfg.inner_max_grad_norm,
    eval_seed=999,
)
plot_adaptation_curve(curve, title="Adaptation curve after meta-training (best checkpoint)")

### Visualize trajectories before/after adaptation

**What to look for:** before adaptation the agent wanders aimlessly; after a few gradient steps it
should head straight toward the (previously hidden) goal.


In [ ]:
def plot_trajectory_pair(
    ep_before: dict,
    ep_after_det: dict,
    task: Point2DTask,
    title: str = "",
    stoch_after: list[dict] | None = None,
):
    goal = task.goal
    plt.figure(figsize=(4.2, 4.2))
    if stoch_after:
        # Light trajectories = multiple *stochastic* rollouts after adaptation (same task).
        # They visualize exploration / variance.
        for j, ep in enumerate(stoch_after):
            if j == 0:
                plt.plot(
                    ep["obs"][:, 0],
                    ep["obs"][:, 1],
                    linewidth=2,
                    alpha=0.6,
                    label="after (stoch sample)",
                )
            else:
                plt.plot(ep["obs"][:, 0], ep["obs"][:, 1], linewidth=1, alpha=0.12)

    plt.plot(
        ep_before["obs"][:, 0],
        ep_before["obs"][:, 1],
        marker="o",
        markersize=3,
        linewidth=1,
        label="before",
    )
    plt.plot(
        ep_after_det["obs"][:, 0],
        ep_after_det["obs"][:, 1],
        marker="o",
        markersize=3,
        linewidth=1,
        label="after (det)",
    )
    plt.scatter([0.0], [0.0], marker="s", label="start")
    plt.scatter([goal[0]], [goal[1]], marker="x", s=90, label="goal")
    plt.axhline(0, linewidth=1)
    plt.axvline(0, linewidth=1)
    plt.xlim(-1.4, 1.4)
    plt.ylim(-1.4, 1.4)
    plt.gca().set_aspect("equal", adjustable="box")
    plt.title(title)
    plt.legend()
    plt.show()


# Visualize a few fixed tasks (fixed seed so plots are reproducible)
rng = np.random.RandomState(2026)
demo_tasks = []
while len(demo_tasks) < 5:
    g = rng.uniform(-1.0, 1.0, size=(2,)).astype(np.float32)
    if np.linalg.norm(g) < 0.6:
        continue  # skip very easy near-origin goals
    demo_tasks.append(Point2DTask(goal=g))
env_kwargs = dict(horizon=25, max_action=0.15, noise_std=0.00)

demo_inner_steps = 6  # run more inner updates for clearer post-adaptation trajectories

for i, task in enumerate(demo_tasks):
    env = Point2DEnv(**env_kwargs)
    theta = policy.get_param_dict()

    # before
    ep0_det = rollout_episode(env, task, policy, params=theta, deterministic_action=True)
    ep0 = rollout_episode(
        env, task, policy, params=theta, deterministic_action=False
    )  # show random-walky behavior
    r0 = float(ep0_det["rew"].sum())

    # adapt
    theta_k = theta
    with temp_seed(1000 + i):
        for _ in range(demo_inner_steps):
            support_eps = [
                rollout_episode(env, task, policy, params=theta_k)
                for _ in range(max(3, cfg.support_episodes))
            ]
            loss = reinforce_loss_from_episodes_torch(
                support_eps, policy, theta_k, gamma=cfg.gamma, return_scale=cfg.return_scale
            )
            theta_k = adapt_params(
                theta_k,
                loss,
                step_size=cfg.inner_lr,
                second_order=False,
                max_grad_norm=cfg.inner_max_grad_norm,
            )

    epk = rollout_episode(env, task, policy, params=theta_k, deterministic_action=True)
    rk = float(epk["rew"].sum())

    stoch_eps = [
        rollout_episode(env, task, policy, params=theta_k, deterministic_action=False)
        for _ in range(5)
    ]

    plot_trajectory_pair(
        ep0,
        epk,
        task,
        title=f"Task {i}: det return {r0:.1f} → {rk:.1f} after {demo_inner_steps} inner steps",
        stoch_after=stoch_eps,
    )

## Side-by-side: Full MAML vs FOMAML

**What to look for:** both methods should achieve positive adaptation gain, but FOMAML is often more
stable while full MAML can be noisier due to higher-variance second-order gradients.


Let's train both algorithms for a longer duration (300 iterations) to compare their performance.

1. **Full MAML** (`second_order=True`): Uses second-order derivatives (Hessian-vector products) for
   the meta-update.
2. **FOMAML** (`second_order=False`): Approximates the meta-gradient by ignoring the Hessian term.

**Expectations:**

- **Theory:** Full MAML computes the correct meta-gradient, so it should technically find a better
  policy or converge with fewer iterations.
- **Practice:** On simple tasks (like this 2D navigation), FOMAML often performs surprisingly close
  to Full MAML. The first-order approximation captures the bulk of the useful signal.
- **Compute:** Full MAML is slower per iteration due to the second-order calculations.

> **Note:** We run for 300 iterations to allow convergence. If running on a slow CPU, you might
> want to reduce `meta_iters`.


In [ ]:
def train_and_score(
    second_order: bool, meta_iters: int = 400, seed: int = 0, *, eval_seed: int = 999
):
    """Train either Full MAML or FOMAML and return a compact score.

    Score = mean deterministic gain (after-before) on a fixed evaluation set.
    """
    init_random(seed=seed)
    pol = GaussianPolicy(obs_dim=2, act_dim=2, hidden=64, action_scale=0.15).to(DEVICE)

    # Use a small rollout budget for speed; early stopping + best-checkpointing handle non-monotonicity.
    cfg2 = MetaConfig(
        meta_iters=meta_iters,
        tasks_per_batch=10,
        inner_steps=3,
        inner_lr=0.05,
        support_episodes=3,
        query_episodes=3,
        meta_lr=4e-4 if not second_order else 1.5e-4,
        meta_lr_second_order=1.5e-4,
        entropy_coef=0.02,
        std_reg_coef=1e-5,
        std_reg_target=-0.7,
        second_order=second_order,
    )

    hist = meta_train(
        pol,
        cfg2,
        goal_range=0.7,
        eval_every=50,
        n_val_tasks=20,
        val_rollouts=1,
        val_seed=123,
        eval_deterministic=True,  # stable checkpoint metric
        early_stop_patience=3,
        early_stop_min_delta=1e-3,
    )

    # Final evaluation on a separate fixed set: adaptation curve (det + stochastic)
    curve = evaluate_adaptation_curve(
        pol,
        n_tasks=30,
        goal_range=0.7,
        k_steps=6,
        inner_lr=cfg2.inner_lr,
        support_episodes=3,
        gamma=cfg2.gamma,
        return_scale=cfg2.return_scale,
        inner_max_grad_norm=cfg2.inner_max_grad_norm,
        eval_seed=eval_seed,
        eval_deterministic=True,
        eval_rollouts=3,
    )

    rets_det = curve["returns_det"]
    gain_det = float(rets_det[cfg2.inner_steps].mean() - rets_det[0].mean())
    return pol, hist, curve, gain_det


pol_maml, hist_maml, curve_maml, gain_maml = train_and_score(True, meta_iters=300, seed=0)
pol_fomaml, hist_fomaml, curve_fomaml, gain_fomaml = train_and_score(False, meta_iters=300, seed=0)

print(f"Full MAML: deterministic mean gain after {3} inner steps = {gain_maml:.2f}")
print(f"FOMAML:    deterministic mean gain after {3} inner steps = {gain_fomaml:.2f}")

plot_training(hist_maml, title_prefix="Full MAML - ")
plot_training(hist_fomaml, title_prefix="FOMAML - ")

plot_adaptation_curve(curve_maml, title="Full MAML: adaptation curve (eval set)")
plot_adaptation_curve(curve_fomaml, title="FOMAML: adaptation curve (eval set)")

In [ ]:
# Optional: multi-seed comparison (set to True to run; this can take a few minutes)
RUN_MULTI_SEED = False

if RUN_MULTI_SEED:
    seeds = [0, 1, 2]
    meta_iters = 250  # keep this modest for a quick multi-seed sanity check

    def _mean_std(xs):
        xs = [float(x) for x in xs]
        m = sum(xs) / len(xs)
        v = sum((x - m) ** 2 for x in xs) / max(1, (len(xs) - 1))
        return m, (v**0.5)

    results = {}
    for name, second_order in [("Full MAML", True), ("FOMAML", False)]:
        gains = []
        for s in seeds:
            _, _, _, g = train_and_score(second_order, meta_iters=meta_iters, seed=s, eval_seed=999)
            gains.append(g)
        m, sd = _mean_std(gains)
        results[name] = (gains, m, sd)

    for name, (gains, m, sd) in results.items():
        gains_str = ", ".join(f"{g:.2f}" for g in gains)
        print(f"{name}: gains = [{gains_str}] | mean ± std = {m:.2f} ± {sd:.2f}")

### Why Full MAML can be noisier in RL (and sometimes look worse than FOMAML)

In supervised meta-learning, full MAML often has an edge because second-order gradients improve the
meta-update direction. In _on-policy RL_, though, the story is different:

- **Compounding variance.** The policy-gradient loss is already a high-variance Monte Carlo estimate
  (trajectory sampling, stochastic actions, long-horizon credit assignment). Full MAML
  backpropagates through the inner update, which effectively differentiates through that noisy
  estimate — compounding the variance. In small-batch RL settings (few rollouts per task), those
  second-order signals can be dominated by noise.
- **FOMAML as a stabilizer.** Dropping the second-order terms (FOMAML) is a biased approximation,
  but it often reduces variance enough to yield comparable or even better validation performance for
  the same compute budget.
- **Hyperparameters matter more.** Inner learning rate, number of support episodes, and return
  normalization can shift which method looks better. If the inner update is already "aggressive,"
  second-order terms can amplify instability.

So if you see FOMAML slightly outperform MAML in this notebook, that's not a sign that MAML is
"wrong" — it reflects the bias/variance tradeoff under noisy RL gradients. In this notebook's
lightweight on-policy setup, FOMAML is often good enough — and simpler.


In [ ]:
# ============================
# Conclusion: quick numerical summary (from this run)
# ============================


def _fmt(x):
    return "N/A" if x is None else f"{float(x):.2f}"


def _summarize_run(name, hist, curve, k_steps: int):
    best_it = hist.get("best_iter", None)
    best_gain = hist.get("best_val_gain_at_best", hist.get("best_val_gain", None))
    best_gain_overall = hist.get("best_val_gain", None)
    best_after = hist.get("best_val_after", None)
    best_before = hist.get("best_val_before", None)

    rets_det = curve["returns_det"]
    gain_det = float(rets_det[k_steps].mean() - rets_det[0].mean())

    print(f"{name}")
    if best_it is not None:
        print(
            f"  best checkpoint: iter {best_it} | val_gain={_fmt(best_gain)} (before={_fmt(best_before)}, after={_fmt(best_after)})"
            + (
                ""
                if best_gain_overall is None
                else f" | best_gain_overall={_fmt(best_gain_overall)}"
            )
        )
    else:
        print("  best checkpoint: N/A")
    print(f"  eval-set deterministic mean gain after {k_steps} inner steps: {gain_det:.2f}")
    print()


_summarize_run("Full MAML", hist_maml, curve_maml, k_steps=3)
_summarize_run("FOMAML", hist_fomaml, curve_fomaml, k_steps=3)

## Conclusion

- **Meta-learning is about fast adaptation.** We meta-train an initialization $\theta$ so that a
  small number of gradient steps on a new task produce adapted parameters $\theta'$ that perform
  much better.
- **Non-monotonic training is normal in RL.** Because our gradients come from on-policy rollouts,
  the meta-objective is noisy. It’s common to see performance improve, peak, and then drift. That’s
  why we checkpoint the best model and use early stopping.
- **Adaptation is measured by a gain curve.** The “adaptation curve” plots return vs. number of
  inner updates. A steep early rise means the learned initialization is easy to adapt.
- **Deterministic vs. stochastic rollouts are complementary.** Stochastic rollouts show exploration
  (often looks like a random walk before adaptation), while deterministic rollouts make it easy to
  see the _direction_ the adapted policy is trying to move.
- **MAML vs. FOMAML.** Full MAML uses a second-order gradient through the inner update; FOMAML drops
  that term. On this toy problem they can be close, but you’ll often see Full MAML learn an
  initialization that adapts a bit more reliably across tasks.

If you want to explore further, try widening the goal distribution (harder tasks), increasing the
number of inner steps, or adding a value-function baseline.


## Notes and ideas to extend this notebook

- **More inner steps**: see how quickly performance saturates.
- **Harder task distributions**:
  - wider goal range,
  - sparse rewards (hard, but instructive for variance and exploration).
- **Baselines / advantage (and why we normalize returns)**: implement a learned value function to
  reduce variance.
- **Meta-RL variants**:
  - RL² (recurrent policy learns to adapt from history),
  - latent-context methods (e.g., infer task embedding).
